# Шаг 2 · Reward Model на CatBoost для KION

Обучаем `E[watched_pct / 100 | прошлая история, кандидат, наблюдавшийся просмотр]`.
Число 0.82 означает прогноз 82% просмотра, **не вероятность клика/выбора фильма**.
Модель учится на реальных взаимодействиях; неизвестные исходы непоказанных/непросмотренных фильмов не превращаются в нули.

Ноутбук уже настроен на `/content/drive/MyDrive/T-lab/dataset`. Все результаты сохраняются на Drive,
в отдельную папку `T-lab/reward_model_v1`. Существующая папка не перезаписывается.
Весь код и компактные характеристики фильмов встроены — дополнительные файлы загружать не нужно.
Характеристики извлечены из исходного KION проекта; SHA256 исходного interactions совпадает с prepared:
`201dc9114eb309da9df6f5f227ada23a371d668edf64a32090fb613a75162ecf`.

**Что делаем:** аудит распределения и пропусков → признаки без будущего → baseline → CatBoost →
MAE/RMSE, калибровка и ранжирование → готовые модель и интерфейс для дальнейшего RL.

> **Версия для репозитория.** Встроенные `base64/zlib`-блоки удалены. Общий feature-store находится в `src/kion_reward_model.py`, а компактный внешний metadata-файл описан в `artifacts/manifest.json`.


In [ ]:
%pip install -q "catboost>=1.2.8,<1.3" "duckdb>=1.4,<2" "pandas>=2.2,<4" "numpy>=1.26,<3" "pyarrow>=17" matplotlib


In [ ]:
from pathlib import Path
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

DATA_DIR = Path('/content/drive/MyDrive/T-lab/dataset')
OUTPUT_DIR = Path('/content/drive/MyDrive/T-lab/reward_model_v1')
WORK_DIR = Path('/content/kion_reward_work')
# Быстрый baseline: детерминированные 500 тыс. train-строк; ВСЯ история и ВСЕ holdout-строки.
# Поставьте None для обучения на всём train; памяти и времени понадобится больше.
TRAIN_MAX_ROWS = 500_000
ITERATIONS = 800
DEPTH = 6
THREADS = 4
MEMORY_LIMIT = '2GB'
SEED = 42


## Протокол и ограничения данных

- Используем прежние календарные границы GPT-2: train до 20 июля, validation 21 июля–5 августа,
  test 6–22 августа 2021 года. Модель подбирается только по validation RMSE, test оценивается после выбора.
- Train: признаки каждого события используют данные **строго до его календарного дня**.
  Validation: общий срез истории на 21 июля; test: на 6 августа (train + validation).
  Внутри периода история не обновляется. Так кандидаты одного пользователя сравниваются по одному контексту.
- Пропуски, бесконечности и значения вне [0,100] исключаются из обучения и оценки как labels.
  Такие события сохраняются в счётчиках истории. Реальные 0 и 100 сохраняются; все исключения считаются отдельно.
- Исторические доли просмотра предполагаются известными к следующему дню. `last_watch_dt` не содержит времени
  показа и момента готовности label; для промышленной оценки нужны эти timestamps.
- Текущие `watched_pct`, `total_dur`, ID строки, user_id и item_id **не входят в признаки**.
  ID используются только для соединений. Категории обрабатываются one-hot, без target encoding.
- Статические характеристики: тип контента, первый жанр, первая страна, год, возрастной рейтинг, число жанров.
  Демографию пользователя не используем: для неё нет истории изменений. Пропуски метаданных отмечаются.
- Reward оценивается на всех допустимых реальных взаимодействиях, дополнительно — на фильмах словаря GPT-2.
  Это другой протокол, чем один target на пользователя при оценке GPT-2; их NDCG напрямую не сравниваются.

Неизвестная политика показа создаёт selection bias. Хорошие offline-метрики ещё не доказывают качество
reward на произвольных рекомендациях GPT-2. Перед RL проверим кандидаты политики и области с малым числом наблюдений.


In [ ]:
import sys, json, shutil, platform, importlib.metadata, gc, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from catboost import CatBoostRegressor, Pool

if OUTPUT_DIR.exists():
    raise FileExistsError(f'{OUTPUT_DIR} уже существует. Выберите новое имя OUTPUT_DIR.')
OUTPUT_DIR.mkdir(parents=True)
WORK_DIR.mkdir(parents=True,exist_ok=True)


In [ ]:
# Общий feature-store хранится как обычный версионируемый Python-модуль.
def find_project_root(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / 'src' / 'kion_reward_model.py').is_file():
            return candidate
    raise FileNotFoundError(
        'Не найден src/kion_reward_model.py. Запускайте ноутбук из клонированного репозитория.'
    )

PROJECT_ROOT = find_project_root()
SRC_DIR = PROJECT_ROOT / 'src'
MODEL_SOURCE_PATH = SRC_DIR / 'kion_reward_model.py'

sys.path.insert(0, str(SRC_DIR))
import kion_reward_model as rm

print('Feature-store module:', MODEL_SOURCE_PATH)


In [ ]:
# Статические признаки каталога вынесены из ноутбука и не публикуются вместе с данными KION.
metadata_candidates = [
    PROJECT_ROOT / 'artifacts' / 'kion_reward_item_features.csv',
    Path('/content/drive/MyDrive/T-lab/reward_model_v1/kion_reward_item_features.csv'),
]
METADATA_PATH = next((path for path in metadata_candidates if path.is_file()), None)
if METADATA_PATH is None:
    raise FileNotFoundError(
        'Не найден kion_reward_item_features.csv. Скопируйте его из reward_model_v1 '
        'в artifacts/ или укажите путь в этой ячейке.'
    )
metadata_path = METADATA_PATH
print('Catalog metadata:', metadata_path)


In [ ]:
required=['train_interactions.csv','validation_interactions.csv','test_interactions.csv',
          'item_mapping.json','split_boundaries.json','input_sha256.json']
local_data=WORK_DIR/'data'
local_data.mkdir(exist_ok=True)
for name in required:
    source=DATA_DIR/name
    if not source.is_file():raise FileNotFoundError(f'Нужен файл: {source}')
    shutil.copy2(source,local_data/name)
boundaries=json.loads((local_data/'split_boundaries.json').read_text(encoding='utf-8'))
mapping=json.loads((local_data/'item_mapping.json').read_text(encoding='utf-8'))
source_identity=json.loads((local_data/'input_sha256.json').read_text(encoding='utf-8'))
assert source_identity['sha256']=='201dc9114eb309da9df6f5f227ada23a371d668edf64a32090fb613a75162ecf', 'Проверьте соответствие встроенных metadata вашему датасету'
input_hashes={n:rm.sha256(local_data/n) for n in required}
db_path=WORK_DIR/'reward_features.duckdb'
if db_path.exists():raise FileExistsError('База предыдущего запуска существует. Задайте новую WORK_DIR.')
store=rm.FeatureStore(db_path,memory_limit=MEMORY_LIMIT,threads=THREADS)
store.ingest(local_data,metadata_path,mapping)
audit,duplicates=store.audit(boundaries)
audit.to_csv(OUTPUT_DIR/'label_audit.csv',index=False)
display(audit)
print('Повторных пар user-item:',duplicates)
print('Пропуски/некорректные labels исключены; нули сохранены.')

distribution=store.con.execute("""SELECT split, floor(watched_pct/10)*10 bin_start,
    count(*) n FROM events WHERE y IS NOT NULL GROUP BY split,bin_start ORDER BY split,bin_start""").df()
distribution.to_csv(OUTPUT_DIR/'target_distribution.csv',index=False)
fig,ax=plt.subplots(figsize=(10,4))
for split,grp in distribution.groupby('split'):
    ax.plot(grp.bin_start,grp.n/grp.n.sum(),marker='o',label=split)
ax.set(xlabel='watched_pct: начало интервала 10 п.п.; 100 отдельно',ylabel='Доля взаимодействий')
ax.legend();fig.tight_layout();fig.savefig(OUTPUT_DIR/'target_distribution.png',dpi=150);plt.show()
train_stats=store.con.execute("SELECT avg(y),median(y),count(y) FROM events WHERE split='train'").fetchone()
train_mean,train_median,train_valid=float(train_stats[0]),float(train_stats[1]),int(train_stats[2])
assert train_valid>0


## Признаки истории

Для пользователя, фильма, жанра и пары пользователь–жанр считаем число прошлых взаимодействий,
среднюю долю просмотра со сглаживанием, долю досмотров ≥80%, долю нулевых просмотров,
давность последнего события и жанровую склонность. У пользователя также есть разброс прошлых оценок.
Сглаживание использует только глобальную статистику, доступную **до cutoff**, а не среднее по будущему train.
Если истории ещё нет, используется нейтральный prior 0.5 и признак отсутствия истории.
Таблицы истории содержат train+validation; строгое ASOF-соединение гарантирует, что будущие строки не попадут в признаки.
Test-labels в таблицы истории не входят.


In [ ]:
start=time.perf_counter()
store.build_histories(boundaries['test']['start'])
feature_paths={}
for split in ['train','validation','test']:
    path=WORK_DIR/f'{split}_features.parquet'
    store.export_features(split,boundaries,path,TRAIN_MAX_ROWS if split=='train' else None)
    feature_paths[split]=path
    print(split,store.con.execute(f'SELECT count(*) FROM read_parquet({rm.sql_path(path)})').fetchone()[0])
print(f'Подготовка признаков: {(time.perf_counter()-start)/60:.1f} мин')
train=pd.read_parquet(feature_paths['train'])
valid=pd.read_parquet(feature_paths['validation'])
assert len(train)>0 and len(valid)>0 and train.y.nunique()>1
assert train.event_day.max()<valid.event_day.min()
assert (train.cutoff==train.event_day).all()
assert (pd.to_datetime(valid.cutoff)==pd.Timestamp(boundaries['validation']['start'])).all()
assert not set(rm.FEATURES)&{'y','watched_pct','total_dur','row_id','user_id','item_id'}
display(pd.DataFrame({'feature':rm.FEATURES,'type':['categorical' if x in rm.CAT_FEATURES else 'numeric' for x in rm.FEATURES]}))
print('Train использовано:',len(train),'из',train_valid,'допустимых строк; все истории сохранены.')
train[['row_id','user_id','item_id','event_day','y']].to_parquet(OUTPUT_DIR/'train_selected_rows.parquet',index=False)


## Обучение

CatBoostRegressor минимизирует RMSE: это соответствует прогнозу ожидаемого значения, нужному для reward.
MAE измеряем отдельно. Ранняя остановка — только по validation RMSE; переобучения на train+validation нет.
Для воспроизводимости используем CPU, фиксированный seed и 4 потока.
При применении прогноз ограничивается [0,1]; в отчёте остаются ошибки и доля выходов за диапазон до ограничения.


In [ ]:
X_train=rm.sanitize_features(train)
X_valid=rm.sanitize_features(valid)
train_pool=Pool(X_train,train.y,cat_features=rm.CAT_FEATURES)
valid_pool=Pool(X_valid,valid.y,cat_features=rm.CAT_FEATURES)
model=CatBoostRegressor(iterations=ITERATIONS,depth=DEPTH,learning_rate=0.05,
    loss_function='RMSE',eval_metric='RMSE',l2_leaf_reg=5,
    random_seed=SEED,thread_count=THREADS,task_type='CPU',
    one_hot_max_size=65535,has_time=True,allow_writing_files=False)
model.fit(train_pool,eval_set=valid_pool,early_stopping_rounds=80,use_best_model=True,verbose=100)
model.save_model(str(OUTPUT_DIR/'reward_model.cbm'))
importance=pd.DataFrame({'feature':rm.FEATURES,'importance':model.get_feature_importance()}).sort_values('importance',ascending=False)
importance.to_csv(OUTPUT_DIR/'feature_importance.csv',index=False)
display(importance.head(15))
del X_train,X_valid,train_pool,valid_pool,train
gc.collect()


## Ошибки и способность ранжировать

Сравниваем с train-mean, train-median, историческими средними фильма/пользователя и их средним.
Исторические baseline имеют тот же cutoff, что и CatBoost.

MAE/RMSE считаются по взаимодействиям; дополнительно — MAE со средним весом каждого пользователя.
Ошибки в процентных пунктах равны ошибке на [0,1] ×100. Для RL важна также калибровка ожидаемого reward.

Для ранжирования берём **только реально просмотренные в holdout фильмы одного пользователя**;
все оцениваем по одному срезу истории. NDCG@5/10 использует линейный gain = watched_pct/100.
Берём пользователей с ≥2 кандидатами и неодинаковыми labels; доля таких пользователей показывается отдельно.
PairAccuracy — доля пар, в которых выше оценён фильм с большим фактическим reward; ties прогноза дают 0.5.
До 256 пар на пользователя: большие группы оцениваются по фиксированной случайной выборке, малые — точно.
`random_expected` — аналитическое ожидание NDCG и top-1 reward случайного порядка.
Это проверка ранжирования наблюдавшихся исходов, **не** оценка рекомендаций по всему каталогу.


In [ ]:
def evaluate(split,frame):
    raw=model.predict(rm.sanitize_features(frame))
    preds={'CatBoost':raw,'train_mean':np.full(len(frame),train_mean),
           'train_median':np.full(len(frame),train_median),
           'item_history':frame.i_reward.to_numpy(),'user_history':frame.u_reward.to_numpy(),
           'user_item_blend':(frame.i_reward.to_numpy()+frame.u_reward.to_numpy())/2}
    rows=[]; ranking=[]
    for scope,keep in [('all_real_interactions',np.ones(len(frame),dtype=bool)),
                       ('gpt2_vocabulary',frame.in_policy_vocab.to_numpy(dtype=bool))]:
        if not keep.any():continue
        f=frame.loc[keep]
        for name,p in preds.items():
            rows.append({'split':split,'scope':scope,'model':name,'n':len(f),
                         **rm.regression_metrics(f.y,p[keep],f.user_id)})
        r,detail=rm.ranking_metrics(f,{name:p[keep] for name,p in preds.items()})
        if not r.empty:
            r['split']=split;r['scope']=scope;ranking.append(r)
            detail.to_parquet(OUTPUT_DIR/f'{split}_{scope}_ranking_per_user.parquet',index=False)
    metrics=pd.DataFrame(rows)
    ranks=pd.concat(ranking,ignore_index=True) if ranking else pd.DataFrame()
    metrics.to_csv(OUTPUT_DIR/f'{split}_regression.csv',index=False)
    ranks.to_csv(OUTPUT_DIR/f'{split}_ranking.csv',index=False)
    cal=rm.calibration_table(frame.y,np.clip(raw,0,1))
    cal.to_csv(OUTPUT_DIR/f'{split}_calibration.csv',index=False)
    fig,ax=plt.subplots(figsize=(5,5));ax.plot([0,1],[0,1],'--',color='gray')
    nonempty=cal.n>0;ax.plot(cal.loc[nonempty,'predicted'],cal.loc[nonempty,'observed'],'o-')
    ax.set(xlabel='Средний прогноз reward',ylabel='Средний реальный reward',title=f'{split}: calibration')
    fig.tight_layout();fig.savefig(OUTPUT_DIR/f'{split}_calibration.png',dpi=150);plt.show()
    prediction_frame=frame[['row_id','user_id','item_id','event_day','cutoff','y','in_policy_vocab','u_missing','i_missing']].copy()
    for name,p in preds.items():prediction_frame[name]=np.clip(p,0,1)
    prediction_frame.to_parquet(OUTPUT_DIR/f'{split}_predictions.parquet',index=False)
    segments=[]
    for segment,mask in [('warm_user',frame.u_missing==0),('cold_user',frame.u_missing==1),
                         ('warm_item',frame.i_missing==0),('cold_item',frame.i_missing==1)]:
        if mask.any():segments.append({'segment':segment,'n':int(mask.sum()),
                          **rm.regression_metrics(frame.y[mask],raw[mask],frame.user_id[mask])})
    pd.DataFrame(segments).to_csv(OUTPUT_DIR/f'{split}_segments.csv',index=False)
    display(metrics[['scope','model','n','MAE','RMSE','MAE_percentage_points','macro_user_MAE']])
    display(ranks)
    return metrics,ranks

valid_metrics,valid_ranks=evaluate('validation',valid)
# Baseline для итогового сравнения выбирается ТОЛЬКО по validation.
scope=valid_metrics.scope=='gpt2_vocabulary'
baselines=valid_metrics.loc[scope & (valid_metrics.model!='CatBoost')]
best_baseline=baselines.sort_values('RMSE').iloc[0].model
print('Сильнейший baseline по validation RMSE:',best_baseline)
del valid;gc.collect()


In [ ]:
# Test используется после завершения fit/выбора числа деревьев. Не подбирайте параметры по этой таблице.
test=pd.read_parquet(feature_paths['test'])
assert (pd.to_datetime(test.cutoff)==pd.Timestamp(boundaries['test']['start'])).all()
test_metrics,test_ranks=evaluate('test',test)
comparison=[]
for split,table in [('validation',valid_metrics),('test',test_metrics)]:
    sub=table[table.scope=='gpt2_vocabulary'].set_index('model')
    comparison.append({'split':split,'baseline':best_baseline,
       'CatBoost_MAE':float(sub.loc['CatBoost','MAE']),
       'baseline_MAE':float(sub.loc[best_baseline,'MAE']),
       'CatBoost_RMSE':float(sub.loc['CatBoost','RMSE']),
       'baseline_RMSE':float(sub.loc[best_baseline,'RMSE']),
       'RMSE_improvement':float(sub.loc[best_baseline,'RMSE']-sub.loc['CatBoost','RMSE'])})
comparison=pd.DataFrame(comparison)
comparison.to_csv(OUTPUT_DIR/'baseline_comparison.csv',index=False)
display(comparison)


In [ ]:
# Сохраняем пригодный для применения артефакт. Test-labels отсутствуют в feature store.
example_user=int(test.user_id.iloc[0])
example_items=test.item_id.head(5).astype(int).tolist()
store.remove_outcomes()
store.close()
shutil.copy2(db_path,OUTPUT_DIR/'reward_features.duckdb')
shutil.copy2(MODEL_SOURCE_PATH, OUTPUT_DIR/'kion_reward_model.py')
shutil.copy2(metadata_path,OUTPUT_DIR/'kion_reward_item_features.csv')
shutil.copy2(local_data/'item_mapping.json',OUTPUT_DIR/'item_mapping.json')
contract={'version':rm.VERSION,'target':'watched_pct/100','prediction_clip':[0,1],
    'history_rule':'events strictly before cutoff calendar day',
    'train_history':'per-event day','validation_cutoff':boundaries['validation']['start'],
    'test_cutoff':boundaries['test']['start'],'history_end_exclusive':boundaries['test']['start'],
    'features':rm.FEATURES,'cat_features':rm.CAT_FEATURES,'boundaries':boundaries,
    'input_sha256':input_hashes,'original_source_sha256':source_identity['sha256'],
    'metadata_sha256':rm.sha256(metadata_path),'module_sha256':rm.sha256(OUTPUT_DIR/'kion_reward_model.py'),
    'model_sha256':rm.sha256(OUTPUT_DIR/'reward_model.cbm'),
    'train_max_rows':TRAIN_MAX_ROWS,'train_valid_labels':train_valid,
    'train_rows_used':min(train_valid,TRAIN_MAX_ROWS) if TRAIN_MAX_ROWS else train_valid,
    'best_iteration':int(model.get_best_iteration()),'baseline_selected_on_validation':best_baseline,
    'parameters':model.get_params(),'seed':SEED,
    'environment':{'python':sys.version,'platform':platform.platform(),
        'packages':{k:importlib.metadata.version(k) for k in ['catboost','duckdb','pandas','numpy']}}}
rm.dump(OUTPUT_DIR/'reward_contract.json',contract)
rm.dump(OUTPUT_DIR/'summary.json',{'comparison':comparison.to_dict('records'),
    'status':'offline_evaluated_not_yet_validated_on_policy_candidates',
    'limitations':['observed-interaction selection bias','day-level label availability assumption',
                  'static catalog metadata','ranking on observed holdout candidates only']})
print('Сохранено на Google Drive:',OUTPUT_DIR)


## Применение к кандидатам GPT-2

Интерфейс принимает **исходные item_id KION**, user_id и cutoff, а историю находит в сохранённых таблицах.
Передавайте токены GPT-2 через `index_to_item_id` из mapping. Никакой текстовый tokenizer не нужен.
Для известного пользователя можно оценивать любые кандидаты по одному фиксированному срезу.
Не добавляйте сгенерированные рекомендации в историю как реальные просмотры с выдуманным watched_pct.

Это baseline с агрегатами истории: он учитывает состав и давность, но не моделирует полный порядок последовательности.
Для новой произвольной истории `[12,83,41]`, которой нет в логах, потребуется отдельное построение пользовательских
признаков; текущий интерфейс не подменяет её историей другого пользователя.
Для новых дат нужно обновить store реальными логами. Веса reward во время первого RL-эксперимента замораживаем.


In [ ]:
reward=rm.RewardModel(OUTPUT_DIR)
try:
    display(reward.predict(example_user,example_items,boundaries['test']['start']))
finally:
    reward.close()
